In [1]:
import pandas as pd
from pathlib import Path

In [2]:
PROCESSED_DIR = Path("../data/processed")
INTERIM_DIR = Path("../data/interim")
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
df = pd.read_csv(PROCESSED_DIR / "atp_matches_1990_onward_with_elo.csv")
df.shape

(112056, 21)

In [4]:
df.head()

,tourney_id,tourney_name,surface,tourney_level,tourney_date,match_num,winner_id,winner_name,winner_age,loser_id,...,loser_age,best_of,winner_rank,winner_rank_points,loser_rank,loser_rank_points,winner_elo,loser_elo,winner_exp,loser_exp
0,1990-339,Adelaide,Hard,A,1990-01-01,1,101843,Sergi Bruguera,18.9,101632,...,20.3,3,25.0,591.0,282.0,70.0,1500.0,1500.0,0.5,0.5
1,1990-339,Adelaide,Hard,A,1990-01-01,2,100714,Broderick Dyke,29.0,101096,...,25.0,3,146.0,157.0,127.0,182.0,1500.0,1500.0,0.5,0.5
2,1990-339,Adelaide,Hard,A,1990-01-01,3,101529,Michael Stich,21.2,101232,...,23.9,3,113.0,215.0,71.0,290.0,1500.0,1500.0,0.5,0.5
3,1990-339,Adelaide,Hard,A,1990-01-01,4,100628,Michiel Schapers,30.2,101434,...,21.9,3,68.0,297.0,80.0,271.0,1500.0,1500.0,0.5,0.5
4,1990-339,Adelaide,Hard,A,1990-01-01,5,101404,Thomas Muster,22.2,101387,...,22.3,3,34.0,496.0,100.0,224.0,1500.0,1500.0,0.5,0.5


In [5]:
df["tourney_date"] = pd.to_datetime(df["tourney_date"])
df = df.sort_values(["tourney_date", "tourney_id", "match_num"]).reset_index(drop=True)

In [6]:
df[["tourney_date", "winner_name", "loser_name", "winner_elo", "loser_elo"]].head()

,tourney_date,winner_name,loser_name,winner_elo,loser_elo
0,1990-01-01,Sergi Bruguera,Per Henricsson,1500.0,1500.0
1,1990-01-01,Broderick Dyke,Gary Muller,1500.0,1500.0
2,1990-01-01,Michael Stich,Nicklas Kroon,1500.0,1500.0
3,1990-01-01,Michiel Schapers,Petr Korda,1500.0,1500.0
4,1990-01-01,Thomas Muster,Claudio Pistolesi,1500.0,1500.0


In [7]:
player_win = pd.DataFrame({
        "tourney_date": df["tourney_date"],
    "tourney_id": df["tourney_id"],
    "tourney_name": df["tourney_name"],
    "surface": df["surface"],
    "tourney_level": df["tourney_level"],
    "best_of": df["best_of"],

    "player_id": df["winner_id"],
    "player_name": df["winner_name"],
    "player_age": df["winner_age"],
    "player_rank": df["winner_rank"],
    "player_rank_points": df["winner_rank_points"],
    "player_elo": df["winner_elo"],

    "opponent_id": df["loser_id"],
    "opponent_name": df["loser_name"],
    "opponent_age": df["loser_age"],
    "opponent_rank": df["loser_rank"],
    "opponent_rank_points": df["loser_rank_points"],
    "opponent_elo": df["loser_elo"],

    "target": 1,
})

In [8]:
player_loss = pd.DataFrame({
    "tourney_date": df["tourney_date"],
    "tourney_id": df["tourney_id"],
    "tourney_name": df["tourney_name"],
    "surface": df["surface"],
    "tourney_level": df["tourney_level"],
    "best_of": df["best_of"],

    "player_id": df["loser_id"],
    "player_name": df["loser_name"],
    "player_age": df["loser_age"],
    "player_rank": df["loser_rank"],
    "player_rank_points": df["loser_rank_points"],
    "player_elo": df["loser_elo"],

    "opponent_id": df["winner_id"],
    "opponent_name": df["winner_name"],
    "opponent_age": df["winner_age"],
    "opponent_rank": df["winner_rank"],
    "opponent_rank_points": df["winner_rank_points"],
    "opponent_elo": df["winner_elo"],

    "target": 0,
})

In [9]:
model_df = pd.concat([player_win, player_loss], ignore_index=True)
model_df = model_df.sort_values(["tourney_date", "tourney_id"]).reset_index(drop=True)
model_df.shape

(224112, 19)

In [10]:
model_df.head()

,tourney_date,tourney_id,tourney_name,surface,tourney_level,best_of,player_id,player_name,player_age,player_rank,player_rank_points,player_elo,opponent_id,opponent_name,opponent_age,opponent_rank,opponent_rank_points,opponent_elo,target
0,1990-01-01,1990-339,Adelaide,Hard,A,3,101843,Sergi Bruguera,18.9,25.0,591.0,1500.0,101632,Per Henricsson,20.3,282.0,70.0,1500.0,1
1,1990-01-01,1990-339,Adelaide,Hard,A,3,100714,Broderick Dyke,29.0,146.0,157.0,1500.0,101096,Gary Muller,25.0,127.0,182.0,1500.0,1
2,1990-01-01,1990-339,Adelaide,Hard,A,3,101529,Michael Stich,21.2,113.0,215.0,1500.0,101232,Nicklas Kroon,23.9,71.0,290.0,1500.0,1
3,1990-01-01,1990-339,Adelaide,Hard,A,3,100628,Michiel Schapers,30.2,68.0,297.0,1500.0,101434,Petr Korda,21.9,80.0,271.0,1500.0,1
4,1990-01-01,1990-339,Adelaide,Hard,A,3,101404,Thomas Muster,22.2,34.0,496.0,1500.0,101387,Claudio Pistolesi,22.3,100.0,224.0,1500.0,1


In [11]:
model_df["elo_diff"] = model_df["player_elo"] - model_df["opponent_elo"]
model_df["age_diff"] = model_df["player_age"] - model_df["opponent_age"]
model_df["rank_diff"] = model_df["opponent_rank"] - model_df["player_rank"]
model_df["rank_points_diff"] = model_df["player_rank_points"] - model_df["opponent_rank_points"]

In [12]:
import numpy as np

In [13]:
model_df["player_rank_log"] = np.log(model_df["player_rank"])
model_df["opponent_rank_log"] = np.log(model_df["opponent_rank"])
model_df["rank_log_diff"] = model_df["opponent_rank_log"] - model_df["player_rank_log"]

In [14]:
FEATURE_COLS = [
    "surface",
    "tourney_level",
    "best_of",
    "player_age",
    "opponent_age",
    "player_rank",
    "opponent_rank",
    "player_rank_points",
    "opponent_rank_points",
    "player_elo",
    "opponent_elo",
    "age_diff",
    "rank_diff",
    "rank_points_diff",
    "rank_log_diff",
    "elo_diff",
]

In [15]:
model_df[FEATURE_COLS + ["target"]].isna().sum()

surface                  106
tourney_level              0
best_of                    0
player_age                33
opponent_age              33
player_rank             4026
opponent_rank           4026
player_rank_points      6368
opponent_rank_points    6368
player_elo                 0
opponent_elo               0
age_diff                  66
rank_diff               6760
rank_points_diff        9096
rank_log_diff           6760
elo_diff                   0
target                     0
dtype: int64

In [16]:
model_df_clean = model_df.dropna(subset=FEATURE_COLS + ["target"]).copy()
model_df_clean.shape

(214922, 26)

In [17]:
model_df_clean[FEATURE_COLS + ["target"]].isna().sum()

surface                 0
tourney_level           0
best_of                 0
player_age              0
opponent_age            0
player_rank             0
opponent_rank           0
player_rank_points      0
opponent_rank_points    0
player_elo              0
opponent_elo            0
age_diff                0
rank_diff               0
rank_points_diff        0
rank_log_diff           0
elo_diff                0
target                  0
dtype: int64

In [18]:
model_df_clean["target"].value_counts()

target
1    107461
0    107461
Name: count, dtype: int64

In [21]:
model_df_clean[[
    "player_name",
    "opponent_name",
    "player_elo",
    "opponent_elo",
    "elo_diff",
    "player_rank",
    "opponent_rank",
    "rank_diff",
    "target",
]].head(1000)

,player_name,opponent_name,player_elo,opponent_elo,elo_diff,player_rank,opponent_rank,rank_diff,target
0,Sergi Bruguera,Per Henricsson,1500.000000,1500.000000,0.000000,25.0,282.0,257.0,1
1,Broderick Dyke,Gary Muller,1500.000000,1500.000000,0.000000,146.0,127.0,-19.0,1
2,Michael Stich,Nicklas Kroon,1500.000000,1500.000000,0.000000,113.0,71.0,-42.0,1
3,Michiel Schapers,Petr Korda,1500.000000,1500.000000,0.000000,68.0,80.0,12.0,1
4,Thomas Muster,Claudio Pistolesi,1500.000000,1500.000000,0.000000,34.0,100.0,66.0,1
...,...,...,...,...,...,...,...,...,...
1275,Ken Flach,Richey Reneberg,1501.239916,1559.738769,-58.498853,254.0,41.0,-213.0,0
1276,Christo Van Rensburg,Malivai Washington,1507.194664,1485.326438,21.868226,30.0,216.0,186.0,0
1277,Diego Perez,Wally Masur,1516.000000,1533.104596,-17.104596,112.0,38.0,-74.0,0
1278,Nicklas Kroon,Jean Philippe Fleurian,1460.826806,1527.885679,-67.058873,78.0,49.0,-29.0,0


In [22]:
model_df.groupby("target")["elo_diff"].mean()

target
0   -69.968939
1    69.968939
Name: elo_diff, dtype: float64

In [23]:
model_df_clean.to_csv(INTERIM_DIR / "model_dataset_v1.csv", index=False)